# LLM-reranker OPE recipe (embedding-MIPS)

You run a **logged reranker** (the current production model) that, for each
query, picks one candidate from a candidate pool and observes a reward
(click / dwell / purchase). You want to evaluate a **new reranker** *offline*,
before A/B testing, using only the logged data and pre-computed embeddings —
**no runtime LLM dependency**.

The natural estimator is **MIPS** (Marginalized IPS): the action space
(candidate pool) is large, but the candidate embeddings carry its structure,
so common support only needs to hold over the *embedding*, not the raw
candidate id.

This recipe runs end-to-end in well under two minutes on CPU.

## 1. Install

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install --quiet skdr-eval scikit-learn

## 2. Synthesize logs with a known ground truth

`make_llm_reranker_synth` builds a deterministic dataset where the reward is a
known linear function of the query·candidate dot product, so the value of the
"sort by dot product" target reranker is closed-form.

In [ ]:
import numpy as np

from skdr_eval.recipes import (
    LLMRerankerLogSchema,
    evaluate_reranker_mips,
    induce_reranker_policy,
    make_llm_reranker_synth,
)

logs, candidate_embeddings, truth = make_llm_reranker_synth(
    n_queries=2000, candidates_per_query=20, embed_dim=32, seed=42
)
logs.head()

In [ ]:
print(f"logging reranker value   V_logging      = {truth.V_logging:.4f}")
print(f"sort-by-dot target value V_sort_by_dot  = {truth.V_sort_by_dot:.4f}")
print(f"candidate pool: {truth.n_candidates} candidates x {truth.embed_dim} dims")

## 3. Validate the log schema

`LLMRerankerLogSchema` documents and enforces the column contract for
LLM-reranker logs. Run it on your own logs to get actionable errors before
evaluation.

In [ ]:
LLMRerankerLogSchema.validate_frame(logs)
print("schema OK:", list(logs.columns))

## 4. Define the candidate (new) reranker and evaluate with MIPS

The candidate reranker scores candidates by the query·candidate dot product
and picks the top one. `evaluate_reranker_mips` assembles the MIPS inputs
(logging propensities, target-policy probabilities, a fitted dot-feature
outcome model) and returns a `DRResult`.

In [ ]:
policy_probs = induce_reranker_policy(logs, candidate_embeddings)
result = evaluate_reranker_mips(logs, candidate_embeddings, policy_probs)

lo = result.V_hat - 2 * result.SE_if
hi = result.V_hat + 2 * result.SE_if
print(f"MIPS V_hat       = {result.V_hat:.4f}  (SE {result.SE_if:.4f})")
print(f"+/- 2 SE band    = [{lo:.4f}, {hi:.4f}]")
print(f"true V_sort_by_dot = {truth.V_sort_by_dot:.4f}")
print(f"ESS = {result.ESS:.1f}   pareto_k = {result.pareto_k:.3f}")

assert lo <= truth.V_sort_by_dot <= hi, "MIPS did not recover V_pi within 2 SE"
print("\nMIPS recovered the target value within +/- 2 SE.")

## 5. Check the embedding-sufficiency diagnostic

MIPS is biased when the embedding is not a sufficient statistic for the
reward. The diagnostic estimates how much action signal the embedding throws
away — a small gap means MIPS is trustworthy here.

In [ ]:
from skdr_eval import embedding_sufficiency_diagnostic

actions = logs["candidate_id"].to_numpy()
# Oracle-free outcome stand-in: predict reward from the observed dot product.
q_hat_obs = np.full(len(logs), float(logs["reward"].mean()))
report = embedding_sufficiency_diagnostic(
    Y=logs["reward"].to_numpy(),
    q_hat=q_hat_obs,
    A=actions,
    action_embedding=candidate_embeddings,
)
print(report.notes)
print(f"action-signal gap = {report.r2_action:.3f}")

## What we learned

- We evaluated a *new* reranker offline, with no LLM at runtime, using only
  logged decisions and pre-computed embeddings.
- MIPS recovered the closed-form target value within ±2 SE.
- Offline evaluation is decision support, **not** a replacement for an online
  A/B test — use the support diagnostics (ESS, Pareto-k, embedding
  sufficiency) to decide whether the estimate is trustworthy enough to act on.